# Module 9: Regression to the Mean

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Anything picked for being extreme tends to look better next time. The
Beginner series asserted that. This module measures it, and finds that on
this dataset it is **weaker than the slogan suggests**, for a reason worth
understanding.

**About 25 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm


def window(agency_id, lo, hi):
    d = f[(f["agency_id"] == agency_id) & (f["year_month"] >= lo)
          & (f["year_month"] < hi)]
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


# Only agencies that received nothing, so any movement is not a program.
early = {a: window(a, "2019-01", "2020-07") for a in COMPARISON}
late = {a: window(a, "2022-07", "2023-07") for a in COMPARISON}
print(f"{len(COMPARISON)} agencies, none of which took the training")

## 2. Do the ones that started high fall further

In [ ]:
rows = []
for a in COMPARISON:
    rows.append({"agency": NAME[a], "2019 to mid 2020": round(early[a], 2),
                 "2022 to mid 2023": round(late[a], 2),
                 "change": f"{100 * (late[a] / early[a] - 1):+.1f}%"})
pd.DataFrame(rows).sort_values("2019 to mid 2020", ascending=False).set_index("agency")

In [ ]:
x = np.array([early[a] for a in COMPARISON])
y = np.array([100 * (late[a] / early[a] - 1) for a in COMPARISON])
z = sm.OLS(y, sm.add_constant(x)).fit()
print(f"  slope: {z.params[1]:.1f} percentage points of change "
      f"per unit of starting rate")
print(f"  p = {z.pvalues[1]:.3f}, R squared = {z.rsquared:.2f}, n = {len(x)}")
print(f"\n  an agency starting one point higher falls "
      f"{-z.params[1]:.1f} points further")

The slope points the expected way and it is **not statistically
distinguishable from zero** with seven agencies. Report it as what it is:
suggestive, underpowered, and in the direction the mechanism predicts.

## 3. How much of the ranking is real

Regression to the mean is driven by the **noisy part** of a measurement. If
an agency's level is measured precisely, there is little noise to revert.

Split the pre period in half and see whether an agency high in one half is
high in the other.

In [ ]:
h1 = {a: window(a, "2019-01", "2021-04") for a in COMPARISON}
h2 = {a: window(a, "2021-04", "2023-07") for a in COMPARISON}
u = np.array([h1[a] for a in COMPARISON])
v = np.array([h2[a] for a in COMPARISON])
print(f"  correlation between the two halves: {np.corrcoef(u, v)[0, 1]:.2f}")
print("\n  agency                              first half   second half")
for a in sorted(COMPARISON, key=lambda k: -h1[k]):
    print(f"  {NAME[a]:34s} {h1[a]:8.2f} {h2[a]:12.2f}")

**A correlation of 0.94.** Over a two year window these agencies' levels are
almost entirely real, not luck. That is why the regression in section 2 is
weak: there is little noise in a two year average for it to work on.

**Regression to the mean is not a property of a population. It is a property
of a measurement.** The same agencies ranked on a single month would revert
sharply, and ranked on a two year average barely at all.

In [ ]:
one_month = {a: window(a, "2021-06", "2021-07") for a in COMPARISON}
next_month = {a: window(a, "2021-07", "2021-08") for a in COMPARISON}
p = np.array([one_month[a] for a in COMPARISON])
q = np.array([next_month[a] for a in COMPARISON])
print(f"  correlation between two adjacent single months: "
      f"{np.corrcoef(p, q)[0, 1]:.2f}")
print(f"  correlation between two two year halves:        "
      f"{np.corrcoef(u, v)[0, 1]:.2f}")

Same agencies, same outcome, two very different answers. **Ranking agencies
on one month is mostly ranking their luck**, and anything selected from that
ranking will revert hard.

## 4. What follows for an evaluation

| If recipients were chosen on | Expect reversion that is | Do |
|---|---|---|
| a single month or quarter | large | never select on one period |
| a year | moderate | use a comparison group chosen the same way |
| several years | small | still use a comparison group |
| something other than the outcome | none from this mechanism | check for other selection |

The comparison group is the fix in every row. **If the comparison agencies
were selected by the same rule, they revert by the same amount and it
subtracts out.** That is a much stronger reason to use one than any of the
arguments in Module 4.

## Exercise

Quantify how large the reversion would be if agencies were ranked on a single
month, by simulating agencies that genuinely never change.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    rng = np.random.default_rng(8)
    means = {a: f[f["agency_id"] == a]["n_uof"].mean() for a in COMPARISON}
    gaps = []
    for _ in range(400):
        m1 = {a: rng.poisson(means[a]) for a in COMPARISON}
        m2 = {a: rng.poisson(means[a]) for a in COMPARISON}
        order = sorted(COMPARISON, key=lambda a: -m1[a])
        top, bottom = order[:3], order[-3:]
        ch = lambda g: np.mean([(m2[a] - m1[a]) / max(m1[a], 1) for a in g])
        gaps.append(100 * (ch(top) - ch(bottom)))
    print("  agencies whose true rate never changes at all, ranked on one month:\n")
    print(f"    the top three then change by {np.mean([g for g in gaps]):+.0f} points")
    print(f"    more than the bottom three, on average")
    print(f"    (median {np.median(gaps):+.0f}, and it is negative in "
          f"{100 * np.mean(np.array(gaps) < 0):.0f} percent of draws)")
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

The simulated agencies never change. Their true rates are fixed for the whole
exercise. And ranked on one month, the top three fall dramatically relative
to the bottom three, in essentially every draw.

**Nothing happened to any of them.** The entire gap is the ranking being
partly a ranking of luck, and luck being redrawn.

Two things follow for real work. Never allocate or evaluate on a single
period ranking, because the reversion will swamp anything a program could do.
And when a funder insists on targeting the worst performers, which is usually
reasonable policy, insist that the comparison group be drawn from the same
ranking so the reversion appears on both sides and cancels.

</details>

---

**Next:** [Module 10: Selection on the Outcome](Module_10_Selection_On_The_Outcome.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*